<a href="https://colab.research.google.com/github/NikitaIvagin/ml-portfolio/blob/main/generative_models/music_generation_chopin_mozart/Music_generation_Chopin_Mozart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y fluidsynth fluid-soundfont-gm 2>/dev/null
!pip install pretty_midi tensorflow -q
!wget -q https://storage.yandexcloud.net/academy.ai/classical-music-midi.zip -O classical-music-midi.zip
!unzip -qo classical-music-midi.zip -d ./dataset 2>/dev/null || true

import os
import numpy as np
import pandas as pd
import pretty_midi
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

np.random.seed(42)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libdouble-conversion3 libevdev2 libfluidsynth3
  libgtk-3-0 libgtk-3-bin libgtk-3-common libgudev-1.0-0 libinput-bin
  libinput10 libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a libqt5dbus5
  libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 librsvg2-common
  libwacom-bin libwacom-common libwacom9 libxcb-icccm4 libxcb-image0
  libxcb-keysyms1 libxcb-render-util0 libxcb-util1 libxcb-xinerama0
  libxcb-xinput0 libxcb-xkb1 libxcomposite1 libxkbcommon-x11-0 libxtst6 qsynth
  qt5-gtk-platformtheme qttranslations5-l10n session-migration
Suggested packages:
  fluid-soundfont-gs gvfs qt5-image-formats-plugins qtwayland5 jackd
The following NEW packages will be installed:
  at-spi2-core fluid-soundfont-gm fluidsynth gsettings-desktop-schemas

In [ ]:
def midi_to_notes(midi_file):
    pm = pretty_midi.PrettyMIDI(midi_file)
    all_notes = []
    for inst in pm.instruments:
        for n in inst.notes:
            all_notes.append((n.pitch, n.start, n.end - n.start))
    all_notes.sort(key=lambda x: x[1])
    result = []
    prev_start = 0.0
    for pitch, start, duration in all_notes:
        step = start - prev_start
        result.append({'pitch': pitch, 'step': step, 'duration': duration})
        prev_start = start
    return pd.DataFrame(result)

In [ ]:
# Шопен для обучения
chopin_notes = []
for f in os.listdir('./dataset/Chopin/'):
    if f.endswith('.mid'):
        try:
            chopin_notes.append(midi_to_notes(os.path.join('./dataset/Chopin/', f)))
        except: pass
chopin_df = pd.concat(chopin_notes, ignore_index=True)

# Моцарт для начальной последовательности генерации
mozart_notes = []
for f in os.listdir('./dataset/Mozart/'):
    if f.endswith('.mid'):
        try:
            mozart_notes.append(midi_to_notes(os.path.join('./dataset/Mozart/', f)))
        except: pass
mozart_df = pd.concat(mozart_notes, ignore_index=True)

print('Шопен, нот:', len(chopin_df))
print('Моцарт, нот:', len(mozart_df))

Шопен, нот: 85355
Моцарт, нот: 68227


In [ ]:
SEQ_LEN = 50

# Нормализация pitch
max_p = chopin_df['pitch'].max()
chopin_df['pitch'] = chopin_df['pitch'] / max_p
scaler = MinMaxScaler()
chopin_df[['step', 'duration']] = scaler.fit_transform(chopin_df[['step', 'duration']])

features = chopin_df[['pitch', 'step', 'duration']].values
X, y = [], []
for i in range(len(features) - SEQ_LEN):
    X.append(features[i:i+SEQ_LEN])
    y.append(features[i+SEQ_LEN])
X = np.array(X)
y = np.array(y)
print('X:', X.shape, 'y:', y.shape)

X: (85305, 50, 3) y: (85305, 3)


In [ ]:
model = Sequential([
    LSTM(256, input_shape=(SEQ_LEN, 3), return_sequences=True),
    Dropout(0.2),
    LSTM(128),
    Dropout(0.2),
    Dense(128, activation='relu'),
    Dropout(0.1),
    Dense(3)
])
model.compile(loss='mse', optimizer='adam')
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50, 256)        │       266,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 480,259 (1.83 MB)

 Trainable params: 480,259 (1.83 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X, y, batch_size=32, epochs=30, verbose=1)

Epoch 1/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 31s 9ms/step - loss: 0.0073
Epoch 2/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0053
Epoch 3/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0051
Epoch 4/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0047
Epoch 5/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0044
Epoch 6/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0041
Epoch 7/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0039
Epoch 8/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0037
Epoch 9/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0035
Epoch 10/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0034
Epoch 11/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0032
Epoch 12/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 41s 9ms/step - loss: 0.0031
Epoch 13/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0030
Epoch 14/30
2666/2666 ━━━━━━━━━━━━━━━━━━━━ 23s 9ms/step - loss: 0.0029
Epoch 15/30
266

In [ ]:
# Увеличение температуры увеличивает случайность параметров ноты
def generate_music(model, seed_seq, n_notes, temperature=0.15):
    current = seed_seq.copy()
    generated = []
    for _ in range(n_notes):
        pred = model.predict(np.expand_dims(current, 0), verbose=0)[0]
        noise = np.random.normal(0, temperature, pred.shape)
        pred = np.clip(pred + noise, 0, 1)
        generated.append(pred)
        current = np.vstack([current[1:], pred])
    return np.array(generated)

# Seed из Моцарта
mozart_df['pitch'] = mozart_df['pitch'] / max_p
mozart_df[['step', 'duration']] = scaler.transform(mozart_df[['step', 'duration']])
mozart_feat = mozart_df[['pitch', 'step', 'duration']].values
if len(mozart_feat) >= SEQ_LEN:
    seed_idx = np.random.randint(0, len(mozart_feat) - SEQ_LEN)
    seed_seq = mozart_feat[seed_idx:seed_idx+SEQ_LEN]
else:
    seed_seq = features[:SEQ_LEN]

In [ ]:
generated = generate_music(model, seed_seq, 100, temperature=0.1)
print('Сгенерировано нот:', len(generated))

Сгенерировано нот: 100


In [ ]:
# Преобразование троек параметров в объект pretty midi
def triples_to_midi(triples, scaler, filename='generated_chopin.mid'):
    pitches = (triples[:, 0] * max_p).astype(int)
    steps, durs = scaler.inverse_transform(triples[:, 1:3]).T
    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=0)
    t = 0.0
    for i in range(len(triples)):
        t += steps[i]
        inst.notes.append(pretty_midi.Note(100, int(np.clip(pitches[i], 0, 127)), t, t + max(0.01, durs[i])))
    pm.instruments.append(inst)
    pm.write(filename)
    return pm

pm = triples_to_midi(generated, scaler)
print('MIDI сохранён в файл generated_chopin.mid')

!fluidsynth -ni /usr/share/sounds/sf2/FluidR3_GM.sf2 generated_chopin.mid -F generated_chopin.wav -r 44100 2>/dev/null
print('Аудио: generated_chopin.wav')

from IPython.display import Audio
display(Audio('generated_chopin.wav'))

MIDI сохранён в файл generated_chopin.mid
FluidSynth runtime version 2.2.5
Copyright (C) 2000-2022 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file 'generated_chopin.wav'..
Аудио: generated_chopin.wav
